# Phase 3: Data Cleaning and Spend Classification

**Goal:** turn messy ERP purchasing data into a trusted, analysis-ready dataset.

Raw procurement data is never clean. Suppliers appear under several names, purchase lines are missing categories, descriptions are typed inconsistently, and a few prices contain data entry errors. Every number in a dashboard or savings estimate is only as good as this step.

**What this notebook does:**

1. Standardizes plant names and units of measure
2. Merges duplicate vendor records into true suppliers
3. Cleans item descriptions and recovers missing material numbers
4. Detects and corrects price entry errors
5. Fills missing categories with a layered approach: material master, then a text classification model, then a supplier-based fallback
6. Validates every step against the answer key
7. Saves clean tables and writes a data quality report

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# Works whether the notebook runs from the project root or the notebooks folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from cleaning import (SITE_MAP, UNIT_MAP, standardize, group_vendors, canonical_names,
                      clean_description, turnaround_tag, recover_material_numbers,
                      flag_price_outliers)

RAW, CLEAN, KEY = ROOT / "data" / "raw", ROOT / "data" / "clean", ROOT / "data" / "answer_key"
CLEAN.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60)

## 1. Load the raw data and profile it

In [2]:
read = lambda name, **kw: pd.read_csv(RAW / f"{name}.csv", **kw)
po = read("po_lines", dtype={"po_number": str})
vm = read("vendor_master")
item_master = read("item_master")
categories = read("categories")

print(f"PO lines: {len(po):,}   vendor records: {len(vm)}   items in master: {len(item_master)}")
profile = pd.DataFrame({
    "missing_%": (po.isna().mean() * 100).round(1),
    "distinct_values": po.nunique(),
})
profile

PO lines: 38,325   vendor records: 99   items in master: 33


,missing_%,distinct_values
po_number,0.0,19218
po_line,0.0,7
site,0.0,25
vendor_id,0.0,99
vendor_name,0.0,98
requisition_date,1.0,1116
po_date,0.0,1096
category_code,14.8,12
material_number,47.6,33
description,0.0,8735


**What we see:** about 15% of lines have no category, many lines have no material number (all service lines by design, plus some materials), and a small share have no requisition date. There are far more distinct descriptions and vendor names than real items and suppliers, which signals inconsistent data entry.

## 2. Standardize plant names and units

In [3]:
print("Raw plant values:", sorted(po["site"].unique()))
po["site_id"] = standardize(po["site"], SITE_MAP)
po["uom"] = standardize(po["unit"], UNIT_MAP)

# Any value not in the mapping becomes NaN, so nothing slips through silently
print("Unmapped plants:", po["site_id"].isna().sum(), "| unmapped units:", po["uom"].isna().sum())
print("Lines with non-standard plant names fixed:", (po["site"] != po["site_id"]).sum())

Raw plant values: ['BATON ROUGE PLANT', 'BR', 'BTR', 'Baton Rouge', 'Baton Rouge Plt', 'CNR', 'CON', 'CONROE PLANT', 'Conroe', 'Conroe Plt', 'FPT', 'FREEPORT PLANT', 'FRP', 'Freeport', 'Freeport Plt', 'L. Charles', 'LAKE CHARLES PLANT', 'LC', 'LCH', 'Lake Charles', 'PAS', 'PASADENA PLANT', 'PSD', 'Pasadena', 'Pasadena TX']
Unmapped plants: 0 | unmapped units: 0
Lines with non-standard plant names fixed: 1531


**Why a mapping table and not fuzzy matching?** There are only five plants and a few units, and abbreviations like "BR" or "FPT" can't be matched by spelling similarity. For small, known lists, an explicit mapping is more reliable and easy to audit.

## 3. Merge duplicate vendor records into true suppliers

The same supplier often exists several times in an ERP vendor master, set up by different people at different times. Two rules link records:

1. **Same tax ID:** the strongest evidence.
2. **Very similar cleaned names in the same state:** after removing punctuation and legal suffixes (Inc, LLC, Co...) and expanding abbreviations, names are compared with a fuzzy matching score (0 to 100). A score of 90 or higher counts as a match.

Linked records are then grouped, so if A matches B and B matches C, all three are one supplier.

In [4]:
vm_grouped = group_vendors(vm, threshold=90)
po_counts = po["vendor_id"].value_counts()
vendors = canonical_names(vm_grouped, po_counts)

print(f"{len(vendors)} vendor records -> {vendors['supplier_id'].nunique()} suppliers")
print(vendors["match_reason"].str.replace(r"_\d+$", "", regex=True).value_counts().to_string())

example = vendors[vendors["supplier_id"] == vendors.loc[vendors["vendor_name"].str.contains("Scaffold Services", case=False), "supplier_id"].iloc[0]]
example[["vendor_id", "vendor_name", "clean_name", "tax_id", "match_reason", "supplier_name"]]

99 vendor records -> 48 suppliers
match_reason
group_anchor    48
tax_id          37
name_match      14


,vendor_id,vendor_name,clean_name,tax_id,match_reason,supplier_name
38,V10039,Gulf Coast Scaffold Services LLC,GULF COAST SCAFFOLD SERVICES,95-8807414,group_anchor,Gulf Coast Scaffold Services LLC
39,V10040,Gulf Coast Scaffold Svcs,GULF COAST SCAFFOLD SERVICES,95-8807414,tax_id,Gulf Coast Scaffold Services LLC
40,V10041,"GUL COAST SCAFFOLD SERVICES, LLC",GUL COAST SCAFFOLD SERVICES,95-8807414,tax_id,Gulf Coast Scaffold Services LLC


**Why both rules?** About 20% of vendor records have no tax ID, so name matching catches duplicates the tax ID rule misses. Requiring the same state prevents false merges between different companies with similar names.

In [5]:
# Validate against the answer key
vendor_key = pd.read_csv(KEY / "vendor_map.csv").rename(columns={"supplier_id": "true_supplier_id"})
check = vendors.merge(vendor_key, on="vendor_id")
impure = (check.groupby("supplier_id")["true_supplier_name"].nunique() > 1).sum()
split = (check.groupby("true_supplier_name")["supplier_id"].nunique() > 1).sum()
print(f"Groups mixing different real suppliers: {impure}")
print(f"Real suppliers split across groups:     {split}")

# Sensitivity: how does name matching alone perform at different thresholds?
for t in (65, 75, 90, 99):
    g = group_vendors(vm.assign(tax_id=None), threshold=t).merge(vendor_key, on="vendor_id")
    print(f"name-only threshold {t}: {g['supplier_group'].nunique()} groups "
          f"(true: {vendor_key['true_supplier_id'].nunique()}), "
          f"mixed groups: {(g.groupby('supplier_group')['true_supplier_name'].nunique() > 1).sum()}")

Groups mixing different real suppliers: 0
Real suppliers split across groups:     0
name-only threshold 65: 42 groups (true: 48), mixed groups: 5
name-only threshold 75: 48 groups (true: 48), mixed groups: 0
name-only threshold 90: 48 groups (true: 48), mixed groups: 0
name-only threshold 99: 54 groups (true: 48), mixed groups: 0


**Reading the threshold test:** at a low threshold (65), different companies with similar names get merged into mixed groups, which would corrupt every supplier-level analysis. At a very high threshold (99), real duplicates with small typos are missed, leaving more groups than true suppliers. Thresholds from about 75 to 97 work well here; 90 leaves a safety margin on both sides. Combining name matching with the tax ID rule adds a second, independent line of evidence.

## 4. Clean descriptions and recover missing material numbers

In [6]:
po["desc_clean"] = po["description"].map(clean_description)
po["turnaround_tag"] = po["description"].map(turnaround_tag)
print(f"Distinct descriptions: {po['description'].nunique():,} raw -> {po['desc_clean'].nunique():,} cleaned")

po[["description", "desc_clean"]].sample(8, random_state=7)

Distinct descriptions: 8,735 raw -> 779 cleaned


,description,desc_clean
4328,UT Thickness Inspection TECH - TA2024,UT THICKNESS INSPECTION TECHNICIAN
12585,PUMP IMPLELER 10IN SS,PUMP IMPLELER 10IN STAINLESS
9745,GLB VLV 4IN CARBON STEEL 150# RF,GLOBE VALVE 4IN CARBON STEEL 150# RF
7875,DIESEL GENERATOR 200KW RENTAL WEEK,DIESEL GENERATOR 200KW RENTAL WEEK
28403,globe vlv 4in cs 150# rf,GLOBE VALVE 4IN CARBON STEEL 150# RF
13789,ELEC MOTOR 50HP TEFC,ELECTRIC MOTOR 50HP TEFC
11768,PUMP MECHANICAL SEAL REPLACEMENT SVC - TA2023,PUMP MECHANICAL SEAL REPLACEMENT SERVICE
21036,HYDROBLASTING UNIT 20K PSI WITH CREW SHIFT SEE ATTACHED,HYDROBLASTING UNIT 20K PSI WITH CREW SHIFT


In [7]:
had_material = po["material_number"].notna()
po["material_number_filled"] = recover_material_numbers(po, item_master, min_score=90)
po["material_source"] = np.select(
    [had_material, po["material_number_filled"].notna()], ["original", "recovered"], default=None)
print(po["material_source"].value_counts(dropna=False).to_string())
print("Remaining lines without a material number are services, which have no material master entry.")

material_source
original     20076
None         16038
recovered     2211
Remaining lines without a material number are services, which have no material master entry.


## 5. Detect and correct price entry errors

Each line's unit price is compared with the median price of comparable lines: the same material, or for services, the same supplier and unit. A price more than 20 times the median is flagged. Prices almost exactly 100 times the median are a classic decimal point slip, so they are corrected; anything else is only flagged for review.

In [8]:
po["supplier_id"] = po["vendor_id"].map(vendors.set_index("vendor_id")["supplier_id"])
po["price_group"] = po["material_number_filled"].fillna(po["supplier_id"])
po["price_flag"] = flag_price_outliers(po, ["price_group", "uom"], ratio=20)

median = po.groupby(["price_group", "uom"])["unit_price"].transform("median")
ratio = po["unit_price"] / median
fixable = po["price_flag"] & ratio.between(50, 200)
po["unit_price_original"] = po["unit_price"]
po.loc[fixable, "unit_price"] = (po.loc[fixable, "unit_price"] / 100).round(2)
po["line_amount_original"] = po["line_amount"]
po["line_amount"] = (po["quantity"] * po["unit_price"]).round(2)

print(f"Flagged: {po['price_flag'].sum()}  corrected: {fixable.sum()}")
print(f"Spend overstated by these errors: ${(po['line_amount_original'] - po['line_amount']).sum():,.0f}")
po.loc[po["price_flag"], ["description", "unit_price_original", "unit_price"]].head()

Flagged: 23  corrected: 22
Spend overstated by these errors: $15,042,670


,description,unit_price_original,unit_price
888,Elbow 90 DEG 4in Sch 40 Carbon Steel,4320.0,43.20
4003,PRESSURE RELIEF VLV 1X2IN,329688.0,3296.88
5172,FITETR OVERTIME,9479.0,94.79
6307,mech seal cartirdge 2.5in,224640.0,2246.40
6532,SERVICES RENDERED,102011.0,1020.11


**Why this matters:** a handful of decimal errors can overstate spend by millions and distort every dashboard and savings estimate built on top. Catching them first protects the credibility of everything that follows.

## 6. Fill missing categories: a layered approach

Categories are filled in order of reliability:

1. **Original category**, when present.
2. **Material master:** if the line has a material number, its category is known for certain.
3. **Text classification model:** for the rest (mostly services), a model reads the description and predicts the category. Only confident predictions (60% or higher) are accepted.
4. **Supplier fallback:** for vague descriptions like "MISC LABOR", the model is unsure, so the line takes its supplier's most common category.

Each line records where its category came from, so the result is auditable.

In [9]:
po["text"] = po["desc_clean"] + " UOM_" + po["uom"]
labeled = po[po["category_code"].notna()]

X_train, X_test, y_train, y_test = train_test_split(
    labeled["text"], labeled["category_code"], test_size=0.2, random_state=42,
    stratify=labeled["category_code"])

# Character n-grams handle typos and abbreviations better than whole words
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
model = LogisticRegression(max_iter=2000, C=5)
model.fit(vectorizer.fit_transform(X_train), y_train)

proba = model.predict_proba(vectorizer.transform(X_test))
pred, conf = model.classes_[proba.argmax(axis=1)], proba.max(axis=1)
print(f"Test accuracy, all lines:              {accuracy_score(y_test, pred):.1%}")
print(f"Share of lines below 60% confidence:   {(conf < 0.6).mean():.1%}")
print(f"Accuracy on confident lines (>= 60%):  {accuracy_score(y_test[conf >= 0.6], pred[conf >= 0.6]):.1%}")

Test accuracy, all lines:              98.3%
Share of lines below 60% confidence:   2.7%
Accuracy on confident lines (>= 60%):  99.9%


In [10]:
# What does the model struggle with? Mostly vague descriptions.
test_view = pd.DataFrame({"text": X_test, "actual": y_test, "predicted": pred, "confidence": conf.round(2)})
test_view.sort_values("confidence").head(8)

,text,actual,predicted,confidence
27983,MATERIALS PER QUOTE UOM_EA,C09,C02,0.3
1018,MATERIALS PER QUOTE UOM_EA,C11,C02,0.3
27181,MATERIALS PER QUOTE UOM_EA,C02,C02,0.3
12666,MATERIALS PER QUOTE UOM_EA,C03,C02,0.3
34809,MATERIALS PER QUOTE UOM_EA,C11,C02,0.3
26971,MATERIALS PER QUOTE UOM_EA,C11,C02,0.3
26267,MATERIALS PER QUOTE UOM_EA,C09,C02,0.3
1139,MATERIALS PER QUOTE UOM_EA,C02,C02,0.3


In [11]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         C01       0.99      1.00      0.99       401
         C02       0.97      0.98      0.98       832
         C03       1.00      0.97      0.98       677
         C04       0.95      1.00      0.97       528
         C05       0.99      0.96      0.97       517
         C06       1.00      0.96      0.98       270
         C07       1.00      0.99      0.99       281
         C08       1.00      0.98      0.99       278
         C09       1.00      0.98      0.99       584
         C10       0.99      0.99      0.99       351
         C11       0.97      0.99      0.98      1505
         C12       1.00      1.00      1.00       304

    accuracy                           0.98      6528
   macro avg       0.99      0.98      0.98      6528
weighted avg       0.98      0.98      0.98      6528



In [12]:
mat_category = item_master.set_index("material_number")["category_code"]
supplier_main_category = labeled.groupby("supplier_id")["category_code"].agg(lambda s: s.mode()[0])

missing = po["category_code"].isna()
po["category_source"] = np.where(missing, None, "original")
po["category_confidence"] = np.where(missing, np.nan, 1.0)

# Layer 2: material master
from_master = missing & po["material_number_filled"].map(mat_category).notna()
po.loc[from_master, "category_code"] = po.loc[from_master, "material_number_filled"].map(mat_category)
po.loc[from_master, ["category_source", "category_confidence"]] = ["material_master", 1.0]

# Layers 3 and 4: model, then supplier fallback
remaining = po["category_code"].isna()
proba = model.predict_proba(vectorizer.transform(po.loc[remaining, "text"]))
pred, conf = model.classes_[proba.argmax(axis=1)], proba.max(axis=1)
confident = conf >= 0.6
idx = po.index[remaining]
po.loc[idx[confident], "category_code"] = pred[confident]
po.loc[idx[confident], "category_source"] = "model"
po.loc[idx, "category_confidence"] = conf.round(3)
fallback = idx[~confident]
po.loc[fallback, "category_code"] = po.loc[fallback, "supplier_id"].map(supplier_main_category)
po.loc[fallback, "category_source"] = "supplier_fallback"

print(po["category_source"].value_counts().to_string())
print("Lines still without a category:", po["category_code"].isna().sum())

category_source
original             32637
material_master       3340
model                 2282
supplier_fallback       66
Lines still without a category: 0


In [13]:
# Validate the filled categories against the answer key
truth = pd.read_csv(KEY / "po_line_truth.csv", dtype={"po_number": str})
v = po.merge(truth[["po_number", "po_line", "true_category"]], on=["po_number", "po_line"])
summary = (v.assign(correct=v["category_code"] == v["true_category"])
             .groupby("category_source")["correct"].agg(lines="size", accuracy="mean"))
summary["accuracy"] = (summary["accuracy"] * 100).round(1).astype(str) + "%"
summary

,lines,accuracy
category_source,,
material_master,3340,100.0%
model,2282,99.7%
original,32637,100.0%
supplier_fallback,66,100.0%


## 7. Other checks: dates and PO cycle time

In [15]:
for col in ("requisition_date", "po_date"):
    po[col] = pd.to_datetime(po[col])
po["cycle_days"] = (po["po_date"] - po["requisition_date"]).dt.days
print("Missing requisition dates (cycle time unknown):", po["requisition_date"].isna().sum())
print("Negative cycle times:", (po["cycle_days"] < 0).sum())
po.groupby("site_id")["cycle_days"].median().rename("median_cycle_days").to_frame().T

Missing requisition dates (cycle time unknown): 395
Negative cycle times: 0


site_id,BTR,CNR,FRP,LCH,PAS
median_cycle_days,4.0,4.0,4.0,4.0,13.0


## 8. Save the clean tables

In [16]:
po_clean = po[[
    "po_number", "po_line", "site_id", "vendor_id", "supplier_id", "requisition_date", "po_date",
    "cycle_days", "category_code", "category_source", "category_confidence",
    "material_number_filled", "material_source", "description", "desc_clean", "turnaround_tag",
    "quantity", "uom", "unit_price", "unit_price_original", "price_flag", "line_amount",
    "contract_id", "currency",
]].rename(columns={"material_number_filled": "material_number"})
po_clean.to_csv(CLEAN / "po_lines.csv", index=False)

# Supplier dimension: one row per real supplier
supplier_dim = (vendors.groupby(["supplier_id", "supplier_name"])
    .agg(city=("city", "first"), state=("state", "first"),
         status=("status", lambda s: "Blocked" if (s == "Blocked").any()
                                    else ("Active" if (s == "Active").any() else "Inactive")),
         blocked_date=("blocked_date", "max"), safety_emr=("safety_emr", "mean"),
         insurance_expiry=("insurance_expiry", "max"), vendor_records=("vendor_id", "count"))
    .reset_index())
supplier_dim["safety_emr"] = supplier_dim["safety_emr"].round(2)
supplier_dim.to_csv(CLEAN / "suppliers.csv", index=False)
vendors[["vendor_id", "vendor_name", "supplier_id", "match_reason", "status"]].to_csv(
    CLEAN / "vendor_xref.csv", index=False)

# Other tables: add supplier_id where useful, otherwise copy as-is
vendor_to_supplier = vendors.set_index("vendor_id")["supplier_id"]
invoices = read("invoices", dtype={"po_number": str})
invoices["supplier_id"] = invoices["vendor_id"].map(vendor_to_supplier)
invoices.to_csv(CLEAN / "invoices.csv", index=False)
contracts = read("contracts")
contracts["supplier_id"] = contracts["vendor_id"].map(vendor_to_supplier)
contracts.to_csv(CLEAN / "contracts.csv", index=False)
for name in ("goods_receipts", "sites", "categories", "item_master", "contract_rates",
             "equipment", "work_orders", "wo_parts"):
    read(name, dtype={"po_number": str}).to_csv(CLEAN / f"{name}.csv", index=False)

print("Saved:", sorted(p.name for p in CLEAN.glob("*.csv")))

Saved: ['categories.csv', 'contract_rates.csv', 'contracts.csv', 'equipment.csv', 'goods_receipts.csv', 'invoices.csv', 'item_master.csv', 'po_lines.csv', 'sites.csv', 'suppliers.csv', 'vendor_xref.csv', 'wo_parts.csv', 'work_orders.csv']


## 9. Write the data quality report

In [17]:
acc = v.assign(c=v["category_code"] == v["true_category"]).groupby("category_source")["c"].mean()
report = f"""# Data Quality Report

Generated by `notebooks/03_cleaning_classification.ipynb`. Covers {len(po):,} purchase order lines, January 2023 to December 2025.

## Issues found and fixed

| Issue | Lines or records affected | Fix |
|---|---|---|
| Plant names typed inconsistently | {(po['site'] != po['site_id']).sum():,} lines | Mapped to standard plant codes |
| Units of measure typed inconsistently | {(po['unit'].str.upper() != po['uom']).sum():,} lines | Mapped to standard units |
| Duplicate vendor records | {len(vendors)} records for {vendors['supplier_id'].nunique()} suppliers | Merged by tax ID and fuzzy name matching (score 90+, same state) |
| Missing material numbers on material lines | {(po['material_source'] == 'recovered').sum():,} lines | Recovered by matching descriptions to the item master |
| Decimal-point price errors | {int(fixable.sum())} lines, overstating spend by ${(po['line_amount_original'] - po['line_amount']).sum():,.0f} | Corrected (price divided by 100); original kept |
| Missing categories | {(po['category_source'] != 'original').sum():,} lines | Filled from material master, a text model, or the supplier's main category |

## Category filling results (validated against the answer key)

| Source | Lines | Accuracy |
|---|---|---|
""" + "\n".join(f"| {s} | {(po['category_source'] == s).sum():,} | {acc[s]:.1%} |" for s in acc.index) + f"""

The text model reached {accuracy_score(y_test, model.classes_[model.predict_proba(vectorizer.transform(X_test)).argmax(1)]):.1%} accuracy on held-out lines. Vague descriptions such as "MISC LABOR" are the main source of uncertainty, which is why low-confidence predictions fall back to the supplier's main category.

## Known remaining limitations

- {po['requisition_date'].isna().sum()} lines have no requisition date, so their PO cycle time is unknown.
- Service lines have no material number by design; price comparisons for services use supplier and unit instead.
- About 20% of vendor records have no tax ID; for these, supplier matching relies on names alone.
- In real data, model accuracy would likely be lower than on synthetic data, so low-confidence lines should go to a human review queue.
"""
(ROOT / "docs" / "data_quality_report.md").write_text(report)
print(report)

# Data Quality Report

Generated by `notebooks/03_cleaning_classification.ipynb`. Covers 38,325 purchase order lines, January 2023 to December 2025.

## Issues found and fixed

| Issue | Lines or records affected | Fix |
|---|---|---|
| Plant names typed inconsistently | 1,531 lines | Mapped to standard plant codes |
| Units of measure typed inconsistently | 2,332 lines | Mapped to standard units |
| Duplicate vendor records | 99 records for 48 suppliers | Merged by tax ID and fuzzy name matching (score 90+, same state) |
| Missing material numbers on material lines | 2,211 lines | Recovered by matching descriptions to the item master |
| Decimal-point price errors | 22 lines, overstating spend by $15,042,670 | Corrected (price divided by 100); original kept |
| Missing categories | 5,688 lines | Filled from material master, a text model, or the supplier's main category |

## Category filling results (validated against the answer key)

| Source | Lines | Accuracy |
|---|---|---|
| mate

## Summary

The data is now trusted and analysis-ready: one record per real supplier, standard plants and units, complete categories with an audit trail, and corrected prices. Next, `src/build_database.py` loads these clean tables into a DuckDB star schema for SQL analysis, and the clean CSVs feed the Power BI dashboard.